### **Timer**

Start: 15:55

In [1]:
import pickle
import pandas as pd
import os
from string import punctuation
import random

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

### **Utility**

In [2]:
stemmer = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()
eng_stop = stopwords.words('english')

### **Preprocessing**

In [3]:
def AlterTag (tag: str):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('R'):
        return 'r'
    return 'n'

def Preprocessing (docx: str):
    tokens = word_tokenize(docx.lower())
    tokens = [tok for tok in tokens if tok not in punctuation]
    tokens = [tok for tok in tokens if tok.isalpha()]
    tokens = [tok for tok in tokens if tok not in eng_stop]
    tokens = [stemmer.stem(tok) for tok in tokens]

    tagged = pos_tag(tokens)

    tokens = [lemmatizer.lemmatize(tok, AlterTag(tag)) for tok, tag in tagged]
    return tokens

### **Training**

In [4]:
def Training():
    data = pd.read_csv('./Dataset/Tweets.csv')
    X = data['text']
    Y = data['airline_sentiment']

    # Feature Extraction
    feats = []

    for text, label in zip(X, Y):
        clean = Preprocessing(text)
        feat = {word: True for word in clean}
        feats.append((feat, label))
    
    # Training
    split = int(len(feats) * 0.8)
    train_data = feats[:split]
    evals_data = feats[split:]
    print('Start Training...')

    model = NaiveBayesClassifier.train(train_data)
    acc = accuracy(model, evals_data)

    print('Model Trained')
    print(f'Accuracy: {acc*100}%')

    # Info
    print('Top 5 Most Informative Features are:')
    model.show_most_informative_features(5)

    # Save
    with open('model.pickle', 'wb') as file:
        pickle.dump(model, file)
    print('Model Saved')

    return model

def Load():
    model: any
    if os.path.exists('model.pickle'):
        with open('model.pickle', 'rb') as file:
            model = pickle.load(file)
        return model
    else:
        model = Training()
        return model

### **Support Function**

In [ ]:
def Enter():
    print('Please Enter to Continue...', end='')
    input('')
    print('')

def Write_Tweet():
    while True:
        print('Please Input your Tweet: ', end='')
        docx = input('')

        if len(docx.split()) <= 5:
            print('Please Input at least 5 Words')
        else:
            return docx


def Analyze_Tweet(model, docx: str):
    if len(docx.split()) <= 5:
        print('Please Enter your Tweet First')
        return ''
    
    tokens = word_tokenize(docx)
    tokens = [tok for tok in tokens if tok not in punctuation]
    tokens = [tok for tok in tokens if tok.isalpha()]

    # POS Tagging
    print("POS Tag:")
    tagged = pos_tag(tokens)

    for i, (word, tag) in enumerate(tagged):
        print(f'{i}. {word} : {tag}')
    
    # Synonyms & Antonyms
    print('Synonyms & Antonyms')
    for word in tokens:
        synsets = wordnet.synsets(word)
        synonyms = []
        antonyms = []

        for sys in synsets:
            for lemma in sys.lemmas():
                synonyms.append(lemma.name())
                for anton in lemma.antonyms():
                    antonyms.append(anton.name())
        
        synonyms = list(set(synonyms))
        antonyms = list(set(antonyms))

        print(F'Word: {word}')
        print('='*(5 + len(word)))
        print('Synonyms:')
        if len(synonyms) == 0:
            print('No Synonyms')
        else:
            for w in synonyms:
                print(f'(+) {w}')
        print('')

        print('Antonyms:')
        if len(antonyms) == 0:
            print('No Antonyms')
        else:
            for w in antonyms:
                print(f'(-) {w}')
        print('')

    # Category
    clean = Preprocessing(docx)
    feats = {word: True for word in clean}

    category = model.classify(feats)
    print(f'The Tweet is classified as {category}')
    Enter()


### **Menu**

In [9]:
def Menu():
    model = Load()
    docx = ''

    while True:
        print('')
        print('1. Write Tweet')
        print('2. Analyze Tweet')
        print('3. End Session')
        cc = input(">> ")
        print('')


        if cc == '1':
            docx = Write_Tweet()
        elif cc == '2':
            Analyze_Tweet(model, docx)
        elif cc == '3':
            print('Alright, Thanks for using our App~~ :)')
            break
        else:
            print('Invalid Input')

In [16]:
Menu()


1. Write Tweet
2. Analyze Tweet
3. End Session

Please Enter your Tweet First

1. Write Tweet
2. Analyze Tweet
3. End Session

Please Input your Tweet: Please Input at least 5 Words
Please Input your Tweet: 
1. Write Tweet
2. Analyze Tweet
3. End Session

POS Tag:
0. Please : VB
1. Input : NNP
2. at : IN
3. least : JJS
4. Words : NNS
Please Enter to Continue...
Synonyms & Antonyms
Word: Please
Synonyms:
(+) delight
(+) please

Antonyms:
(-) displease

Word: Input
Synonyms:
(+) remark
(+) input
(+) input_signal
(+) stimulation
(+) stimulus
(+) stimulant
(+) comment

Antonyms:
No Antonyms

Word: at
Synonyms:
(+) astatine
(+) at
(+) atomic_number_85
(+) At

Antonyms:
No Antonyms

Word: least
Synonyms:
(+) to_the_lowest_degree
(+) least

Antonyms:
(-) most

Word: Words
Synonyms:
(+) wrangle
(+) word_of_honor
(+) Word_of_God
(+) articulate
(+) parole
(+) Scripture
(+) countersign
(+) language
(+) watchword
(+) Logos
(+) Christian_Bible
(+) password
(+) phrase
(+) Holy_Writ
(+) discussion
(